<h1 style="color:DodgerBlue">Индивидуальный проект</h1>

**Студент:** Решетова Арина, группа СМАРТБ-25-1

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

Информационная система для учета заказов в компании

### Вариант задания 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Консольное приложение, которое моделирует работу компании по учету заказов. В программе есть классы Company (компания), Employee (сотрудник), Customer (заказчик) и Order (заказ). Сотрудник и заказчик наследуются от базового класса Person. Для хранения данных используются коллекции List<T>, статусы заказа хранятся в enum OrderStatus. При смене статуса заказа вызывается событие, а через делегат сотруднику приходит уведомление.

#### Дополнительное задание
К существующим классам (базовому и производным) добавлены атрибуты и методы (например, Position, OrdersProcessed, PhoneNumber, ChangePosition, ChangePhone, AddProcessedOrder, Notify), в проекте использованы коллекции, делегаты, события и лямбда-выражения.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

Сначала описываем классы:

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// статусы заказа
public enum OrderStatus { Created, InProgress, Completed }

// делегат для уведомл. об изменении статуса
public delegate void StatusChangedHandler(Order order, OrderStatus oldStatus, OrderStatus newStatus);

// базовый класс (наследование)
public class Person
{
    public string FirstName { get; set; }
    public string LastName { get; set; }

    public Person(string firstName, string lastName)
    {
        FirstName = firstName;
        LastName = lastName;
    }

    public string GetFullName()
    {
        return $"{FirstName} {LastName}";
    }

    public virtual void ShowInfo()
    {
        Console.WriteLine($"Имя: {GetFullName()}");
    }
}

// сотрудник
public class Employee : Person
{
    public string Position { get; set; }
    public int OrdersProcessed { get; private set; }   // меняется только внутри класса

    public Employee(string firstName, string lastName, string position)
        : base(firstName, lastName)
    {
        Position = position;
        OrdersProcessed = 0;
    }

    public void AddProcessedOrder()
    {
        OrdersProcessed++;
    }

    public void ChangePosition(string newPosition)
    {
        Position = newPosition;
    }

    // сюда приходит уведомл.
    public void Notify(string message)
    {
        Console.WriteLine($"  [Уведомл. для {GetFullName()}] {message}");
    }

    public override void ShowInfo()
    {
        Console.WriteLine($"Сотрудник: {GetFullName()}, должность: {Position}, обраб. заказов: {OrdersProcessed}");
    }
}

// заказчик
public class Customer : Person
{
    public string PhoneNumber { get; private set; }

    public Customer(string firstName, string lastName, string phoneNumber)
        : base(firstName, lastName)
    {
        PhoneNumber = phoneNumber;
    }

    public void ChangePhone(string newPhone)
    {
        PhoneNumber = newPhone;
    }

    public bool HasPhone()
    {
        return !string.IsNullOrEmpty(PhoneNumber);
    }

    public override void ShowInfo()
    {
        Console.WriteLine($"Заказчик: {GetFullName()}, тел.: {PhoneNumber}");
    }
}

// заказ
public class Order
{
    private static int _nextId = 1;   // счетчик для уник. id

    public int OrderId { get; private set; }
    public string Description { get; set; }
    public Customer Customer { get; private set; }
    public Employee AssignedEmployee { get; private set; }
    public OrderStatus Status { get; private set; }
    public DateTime CreatedAt { get; private set; }

    // событие при смене статуса
    public event StatusChangedHandler StatusChanged;

    public Order(string description, Customer customer)
    {
        OrderId = _nextId++;      // id назначается авто
        Description = description;
        Customer = customer;
        Status = OrderStatus.Created;
        CreatedAt = DateTime.Now;
    }

    public void Assign(Employee employee)
    {
        AssignedEmployee = employee;
    }

    public void ChangeStatus(OrderStatus newStatus)
    {
        if (newStatus == Status) return;   // статус тот же - ничего не делаем
        OrderStatus old = Status;
        Status = newStatus;
        StatusChanged?.Invoke(this, old, newStatus);   // вызываем событие
    }

    public static string GetStatusText(OrderStatus status)
    {
        switch (status)
        {
            case OrderStatus.Created: return "Создан";
            case OrderStatus.InProgress: return "В работе";
            default: return "Завершен";
        }
    }

    public void ShowInfo()
    {
        string emp = AssignedEmployee == null ? "не назначен" : AssignedEmployee.GetFullName();
        Console.WriteLine($"Заказ №{OrderId}: {Description} | заказч.: {Customer.GetFullName()} | сотруд.: {emp} | статус: {GetStatusText(Status)} | {CreatedAt:dd.MM.yyyy}");
    }
}

// компания
public class Company
{
    public string Name { get; set; }
    private List<Employee> _employees = new List<Employee>();
    private List<Order> _orders = new List<Order>();
    private List<Customer> _customers = new List<Customer>();

    public Company(string name)
    {
        Name = name;
    }

    public void AddEmployee(Employee employee)
    {
        _employees.Add(employee);
    }

    public void AddCustomer(Customer customer)
    {
        _customers.Add(customer);
    }

    // создаем заказ и подписываемся на его событие
    public Order CreateOrder(string description, Customer customer)
    {
        Order order = new Order(description, customer);
        order.StatusChanged += OnStatusChanged;
        _orders.Add(order);
        return order;
    }

    // назначаем сотруд. на заказ
    public void AssignEmployee(int orderId, Employee employee)
    {
        Order order = _orders.FirstOrDefault(o => o.OrderId == orderId);
        if (order == null)
        {
            Console.WriteLine("Заказ не найден");
            return;
        }
        order.Assign(employee);
    }

    public void ChangeStatus(int orderId, OrderStatus newStatus)
    {
        Order order = _orders.FirstOrDefault(o => o.OrderId == orderId);
        if (order == null)
        {
            Console.WriteLine("Заказ не найден");
            return;
        }
        order.ChangeStatus(newStatus);
    }

    // обработчик события
    private void OnStatusChanged(Order order, OrderStatus oldStatus, OrderStatus newStatus)
    {
        if (order.AssignedEmployee == null)
        {
            Console.WriteLine($"  Заказ №{order.OrderId} сменил статус, но сотруд. не назначен");
            return;
        }

        // если заказ завершен - +1 к обраб. заказам
        if (newStatus == OrderStatus.Completed)
        {
            order.AssignedEmployee.AddProcessedOrder();
        }

        order.AssignedEmployee.Notify($"заказ №{order.OrderId}: {Order.GetStatusText(oldStatus)} -> {Order.GetStatusText(newStatus)}");
    }

    public void ShowOrdersByStatus(OrderStatus status)
    {
        Console.WriteLine($"--- Заказы со статусом \"{Order.GetStatusText(status)}\" ---");
        var list = _orders.Where(o => o.Status == status).ToList();
        if (list.Count == 0) Console.WriteLine("Нет заказов");
        foreach (var o in list) o.ShowInfo();
    }

    public void FindOrdersByCustomer(string lastName)
    {
        Console.WriteLine($"--- Заказы заказчика {lastName} ---");
        var list = _orders.Where(o => o.Customer.LastName == lastName).ToList();
        if (list.Count == 0) Console.WriteLine("Нет заказов");
        foreach (var o in list) o.ShowInfo();
    }

    // отчет по сотруд.
    public void ShowReport()
    {
        Console.WriteLine($"=== Отчет по сотруд. компании {Name} ===");
        foreach (var e in _employees) e.ShowInfo();
    }
}


Теперь проверяем работу программы:

In [ ]:
// создаем компанию
Company company = new Company("ООО Вектор");

// сотрудники
Employee emp1 = new Employee("Иван", "Петров", "Менеджер");
Employee emp2 = new Employee("Мария", "Сидорова", "Дизайнер");
company.AddEmployee(emp1);
company.AddEmployee(emp2);

// заказчики
Customer cust1 = new Customer("Анна", "Смирнова", "+7 900 111-22-33");
Customer cust2 = new Customer("Олег", "Кузнецов", "+7 900 444-55-66");
company.AddCustomer(cust1);
company.AddCustomer(cust2);

// заказы
Order o1 = company.CreateOrder("Разработка сайта", cust1);
Order o2 = company.CreateOrder("Логотип для кафе", cust2);
Order o3 = company.CreateOrder("Баннер для рекламы", cust1);

// назначаем сотруд.
company.AssignEmployee(o1.OrderId, emp1);
company.AssignEmployee(o2.OrderId, emp2);
company.AssignEmployee(o3.OrderId, emp2);

// меняем статусы (тут срабатывают события и уведомл.)
Console.WriteLine("Меняем статусы:");
company.ChangeStatus(o1.OrderId, OrderStatus.InProgress);
company.ChangeStatus(o1.OrderId, OrderStatus.Completed);
company.ChangeStatus(o2.OrderId, OrderStatus.InProgress);
company.ChangeStatus(o3.OrderId, OrderStatus.InProgress);
company.ChangeStatus(o3.OrderId, OrderStatus.Completed);
Console.WriteLine();

// просмотр по статусам
company.ShowOrdersByStatus(OrderStatus.InProgress);
company.ShowOrdersByStatus(OrderStatus.Completed);
Console.WriteLine();

// поиск по заказчику
company.FindOrdersByCustomer("Смирнова");
Console.WriteLine();

// отчет
company.ShowReport();
